In [10]:
# =====================================================
# 1. BASIC SELF-ATTENTION
# =====================================================

import tensorflow as tf
from tensorflow.keras.layers import Dense

class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super().__init__()
        self.query = Dense(units)
        self.key = Dense(units)
        self.value = Dense(units)

    def call(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = tf.matmul(Q, K, transpose_b=True)
        weights = tf.nn.softmax(scores, axis=-1)

        return tf.matmul(weights, V)

x = tf.random.normal((2, 10, 32))
attention = SelfAttention(32)
output = attention(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)


Input shape : (2, 10, 32)
Output shape: (2, 10, 32)


In [11]:
# =====================================================
# 2. SIMPLE TRANSFORMER ENCODER
# =====================================================

from tensorflow.keras.layers import (
    Input, Embedding, MultiHeadAttention,
    GlobalAveragePooling1D, Dense
)
from tensorflow.keras.models import Model

inputs = Input(shape=(100,))
x = Embedding(10000, 32)(inputs)

attention = MultiHeadAttention(
    num_heads=2,
    key_dim=32
)(x, x)

x = GlobalAveragePooling1D()(attention)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 100, 32)   │    320,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 32)   │      8,416 │ embedding_2[0][0… │
│ (MultiHeadAttentio… │                   │            │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         33 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 328,449 (1.25 MB)

 Trainable params: 328,449 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# =====================================================
# 3. PRE-TRAINED BERT - INFERENCE
# =====================================================

# Install first:
# pip install transformers torch

from transformers import pipeline

classifier = pipeline("sentiment-analysis")

texts = [
    "I love this movie.",
    "This movie is terrible."
]

results = classifier(texts)

for text, result in zip(texts, results):
    print(text, "->", result)


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

I love this movie. -> {'label': 'POSITIVE', 'score': 0.9998736381530762}
This movie is terrible. -> {'label': 'NEGATIVE', 'score': 0.9997261166572571}
